In [0]:
# =============================================================================
# EA Real-Time Measure Register
# =============================================================================
# Notebook:   09_ea_rt_measure_register.py
# Schema:     prd_dash_lab.flood_forecasting_unrestricted
# Table:      ea_rt_measure_register
# Source:     ea_rt_readings_bronze (built by 08_ea_rt_archive_backfill.py)
#             EA Real-Time Flood Monitoring API (station metadata enrichment)
#             https://environment.data.gov.uk/flood-monitoring/id/stations
# Run:        Once after the backfill completes. Subsequently called by
#             10_ea_rt_daily_archive.py to register any new measures.
#
# Purpose:
#   Builds a register of every distinct measure present in the Bronze table.
#   One row per measure URI.
#
#   Two sources combine to produce each row:
#
#   1. The Bronze table itself -- provides the measure URI and all fields
#      carried inline in the readings-full CSV: station_uri, station_reference,
#      label, parameter, qualifier, datum_type, period, unit_name, value_type.
#
#   2. The EA station API -- called once per unique station URI to enrich with
#      fields absent from the CSV: lat, long, easting, northing, grid_reference,
#      ea_region_name, ea_area_name, river_name, catchment_name, town, datum.
#      The station label from the API is also used in preference to the CSV
#      label, which is truncated in some cases (e.g. "AVON WEI" vs "AVON WEIR").
#
# On subsequent runs (called from 10_ea_rt_daily_archive.py):
#   Only measure URIs absent from the existing register are processed.
#   The station API is called only for station URIs not already in the register.
#   Existing rows are never overwritten -- first_seen is preserved.
#
# Data quality notes:
#   - The station API returns inconsistent types for coordinate fields.
#     Some records return grid references (e.g. "SS2137703814") in the northing
#     field, and mixed int/float types across lat, long, easting, northing.
#     All five numeric fields are coerced to float64 before the Spark conversion;
#     unparseable values become null.
#   - Some stations return their label as a list rather than a scalar. The first
#     element is used.
#   - The period field can contain strings (e.g. "m3/s") for some flow measures
#     and is stored as StringType throughout.
#   - Rainfall stations return "Rainfall station" universally. EA policy.
#
# Attribution (OGL):
#   "this uses Environment Agency flood and river level data from the
#    real-time data API (Beta)"
# =============================================================================

import requests
import time
from datetime import datetime, timezone

import pandas as pd

from pyspark.sql import functions as F

# =============================================================================
# CONFIGURATION
# =============================================================================

CATALOG        = "prd_dash_lab"
SCHEMA         = "flood_forecasting_unrestricted"
BRONZE_TABLE   = "ea_rt_readings_bronze"
REGISTER_TABLE = "ea_rt_measure_register"

FULL_BRONZE_NAME   = f"{CATALOG}.{SCHEMA}.{BRONZE_TABLE}"
FULL_REGISTER_NAME = f"{CATALOG}.{SCHEMA}.{REGISTER_TABLE}"

# Throttle between station API calls.
# At ~3,000 unique stations this adds ~15 minutes to the initial run.
# Subsequent runs only call the API for genuinely new stations.
API_CALL_DELAY_SECONDS = 0.3

# Request timeout per station API call
REQUEST_TIMEOUT = 15


In [0]:

# =============================================================================
# STEP 1: EXTRACT DISTINCT MEASURES FROM BRONZE
# =============================================================================
# Pull one row per measure URI, taking the first observed value for each
# inline metadata field. Values are consistent across readings for a given
# measure so first() is safe.

print(f"Extracting distinct measures from {FULL_BRONZE_NAME}...")

sdf_measures = spark.sql(f"""
    SELECT
        measure_uri,
        first(station_uri)       AS station_uri,
        first(station_reference) AS station_reference,
        first(label)             AS csv_label,
        first(parameter)         AS parameter,
        first(qualifier)         AS qualifier,
        first(datum_type)        AS datum_type,
        first(period)            AS period,
        first(unit_name)         AS unit_name,
        first(value_type)        AS value_type
    FROM {FULL_BRONZE_NAME}
    WHERE measure_uri IS NOT NULL
    GROUP BY measure_uri
""")

pdf_measures = sdf_measures.toPandas()
print(f"Distinct measures in Bronze: {len(pdf_measures):,}")


In [0]:
# =============================================================================
# STEP 2: IDENTIFY MEASURES NOT YET IN THE REGISTER
# =============================================================================
# On first run the register does not exist -- process everything.
# On subsequent runs only process measures absent from the register.

register_exists = spark.catalog.tableExists(FULL_REGISTER_NAME)

if register_exists:
    existing_uris = {
        row["measure_uri"]
        for row in spark.sql(f"""
            SELECT measure_uri FROM {FULL_REGISTER_NAME}
        """).collect()
    }
    print(f"Register exists. {len(existing_uris):,} measures already registered.")
    pdf_new = pdf_measures[~pdf_measures["measure_uri"].isin(existing_uris)].copy()
    print(f"New measures to register: {len(pdf_new):,}")
else:
    existing_uris = set()
    pdf_new = pdf_measures.copy()
    print(f"Register does not exist. Processing all {len(pdf_new):,} measures.")

if pdf_new.empty:
    print("Nothing to register.")
    dbutils.notebook.exit("success")


In [0]:
# =============================================================================
# STEP 3: IDENTIFY UNIQUE STATION URIS NEEDING API ENRICHMENT
# =============================================================================
# One API call per unique station URI, not one per measure.
# Reuse enrichment for stations already present in the register.

if register_exists:
    pdf_existing_stations = spark.sql(f"""
        SELECT DISTINCT
            station_uri, station_label, river_name, catchment_name, town,
            lat, long, easting, northing, grid_reference,
            ea_region_name, ea_area_name, datum
        FROM {FULL_REGISTER_NAME}
        WHERE station_uri IS NOT NULL
    """).toPandas()
    known_station_uris = set(pdf_existing_stations["station_uri"])
else:
    pdf_existing_stations = pd.DataFrame()
    known_station_uris    = set()

new_station_uris = set(pdf_new["station_uri"].dropna()) - known_station_uris
print(f"Unique stations needing API enrichment: {len(new_station_uris):,}")



In [0]:
# =============================================================================
# STEP 4: FETCH STATION METADATA FROM THE API
# =============================================================================
# Failed calls are logged with null fields. Use the optional retry cell
# below to re-attempt failures before writing the register.

def fetch_station_metadata(station_uri, timeout):
    """
    Calls the EA station endpoint for a given station URI.
    Returns a dict of enrichment fields, or a dict of Nones on failure.
    """
    empty = {
        "station_uri":    station_uri,
        "station_label":  None,
        "river_name":     None,
        "catchment_name": None,
        "town":           None,
        "lat":            None,
        "long":           None,
        "easting":        None,
        "northing":       None,
        "grid_reference": None,
        "ea_region_name": None,
        "ea_area_name":   None,
        "datum":          None,
    }

    try:
        response = requests.get(station_uri, timeout=timeout)
        response.raise_for_status()
        items = response.json().get("items", {})

        if isinstance(items, list):
            if not items:
                return empty
            items = items[0]

        # label can be returned as a list for some stations -- take first element
        label = items.get("label")
        if isinstance(label, list):
            label = label[0] if label else None

        return {
            "station_uri":    station_uri,
            "station_label":  label,
            "river_name":     items.get("riverName"),
            "catchment_name": items.get("catchmentName"),
            "town":           items.get("town"),
            "lat":            items.get("lat"),
            "long":           items.get("long"),
            "easting":        items.get("easting"),
            "northing":       items.get("northing"),
            "grid_reference": items.get("gridReference"),
            "ea_region_name": items.get("eaRegionName"),
            "ea_area_name":   items.get("eaAreaName"),
            "datum":          items.get("datum"),
        }

    except Exception as e:
        print(f"  WARNING: API call failed for {station_uri}: {e}")
        return empty


# Fetch all new stations
station_enrichments = []
failed_uris         = []
total               = len(new_station_uris)

for i, uri in enumerate(sorted(new_station_uris)):
    if (i + 1) % 100 == 0 or i == 0:
        print(f"  Fetching station {i+1}/{total}...")
    result = fetch_station_metadata(uri, REQUEST_TIMEOUT)
    station_enrichments.append(result)
    if all(result[k] is None for k in result if k != "station_uri"):
        failed_uris.append(uri)
    time.sleep(API_CALL_DELAY_SECONDS)

print(f"\nStation API calls complete.")
print(f"  Successful: {total - len(failed_uris):,}")
print(f"  Failed:     {len(failed_uris):,}")

if failed_uris:
    print("\nFailed URIs (likely timeouts -- use the retry cell below):")
    for uri in failed_uris:
        print(f"  {uri}")

# Combine new stations with those already in the register
pdf_new_stations = pd.DataFrame(station_enrichments)

if not pdf_existing_stations.empty:
    pdf_all_stations = pd.concat(
        [pdf_existing_stations, pdf_new_stations], ignore_index=True
    )
else:
    pdf_all_stations = pdf_new_stations

# Coerce coordinate and datum fields to float64 before the merge.
# The API returns inconsistent types: grid references, integers, and floats
# can all appear in coordinate fields for malformed station records.
# errors='coerce' converts anything unparseable to NaN (null in Spark).
for col in ["lat", "long", "easting", "northing", "datum"]:
    if col in pdf_all_stations.columns:
        pdf_all_stations[col] = (
            pd.to_numeric(pdf_all_stations[col], errors="coerce")
            .astype(float)
        )


In [0]:
# =============================================================================
# OPTIONAL: RETRY FAILED STATION FETCHES
# =============================================================================
# If Step 4 produced failures, paste the failed URIs into the list below
# and run this cell before continuing to Step 5.
# Do not re-run Step 4 -- it repeats all API calls from scratch.

retry_uris = [
    # Paste failed URIs here, one per line, e.g.:
    # "http://environment.data.gov.uk/flood-monitoring/id/stations/XXXX",
]

if retry_uris:
    print(f"Retrying {len(retry_uris)} failed stations...")
    retry_results = [fetch_station_metadata(uri, REQUEST_TIMEOUT) for uri in retry_uris]
    retry_pdf     = pd.DataFrame(retry_results)

    # Coerce numeric fields on the retry results
    for col in ["lat", "long", "easting", "northing", "datum"]:
        if col in retry_pdf.columns:
            retry_pdf[col] = pd.to_numeric(retry_pdf[col], errors="coerce").astype(float)

    # Replace the failed rows with retry results
    pdf_all_stations = pdf_all_stations[
        ~pdf_all_stations["station_uri"].isin(retry_uris)
    ]
    pdf_all_stations = pd.concat(
        [pdf_all_stations, retry_pdf], ignore_index=True
    )
    print("Retry complete. Continue from Step 5.")
else:
    print("No retry URIs specified. Continue from Step 5.")

In [0]:
# =============================================================================
# STEP 5: JOIN STATION ENRICHMENT ONTO THE NEW MEASURES
# =============================================================================

pdf_register = pdf_new.merge(pdf_all_stations, on="station_uri", how="left")

# Use the API station label in preference to the CSV label.
# Fall back to the CSV label if the API returned nothing.
pdf_register["station_label"] = pdf_register["station_label"].fillna(
    pdf_register["csv_label"]
)

# station_label is now the canonical name; csv_label is no longer needed
pdf_register = pdf_register.drop(columns=["csv_label"])

print(f"Register rows to write: {len(pdf_register):,}")


In [0]:
# =============================================================================
# STEP 6: ADD AUDIT COLUMNS
# =============================================================================

now_utc = datetime.now(timezone.utc)

pdf_register["is_active"]           = True
pdf_register["first_seen"]          = now_utc
pdf_register["register_updated_at"] = now_utc


In [0]:
# =============================================================================
# STEP 7: DEFINE THE REGISTER SCHEMA
# =============================================================================
# Coordinate fields are DoubleType throughout -- the API returns inconsistent
# numeric types and easting/northing can arrive as floats after coercion.
# period is StringType -- some flow measures return strings (e.g. "m3/s").

from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, BooleanType, TimestampType
)

register_schema = StructType([
    StructField("measure_uri",         StringType(),    nullable=False),
    StructField("station_uri",         StringType(),    nullable=True),
    StructField("station_reference",   StringType(),    nullable=True),
    StructField("station_label",       StringType(),    nullable=True),
    StructField("river_name",          StringType(),    nullable=True),
    StructField("catchment_name",      StringType(),    nullable=True),
    StructField("town",                StringType(),    nullable=True),
    StructField("parameter",           StringType(),    nullable=True),
    StructField("qualifier",           StringType(),    nullable=True),
    StructField("datum_type",          StringType(),    nullable=True),
    StructField("datum",               DoubleType(),    nullable=True),
    StructField("period",              StringType(),    nullable=True),
    StructField("unit_name",           StringType(),    nullable=True),
    StructField("value_type",          StringType(),    nullable=True),
    StructField("lat",                 DoubleType(),    nullable=True),
    StructField("long",                DoubleType(),    nullable=True),
    StructField("easting",             DoubleType(),    nullable=True),
    StructField("northing",            DoubleType(),    nullable=True),
    StructField("grid_reference",      StringType(),    nullable=True),
    StructField("ea_region_name",      StringType(),    nullable=True),
    StructField("ea_area_name",        StringType(),    nullable=True),
    StructField("is_active",           BooleanType(),   nullable=False),
    StructField("first_seen",          TimestampType(), nullable=False),
    StructField("register_updated_at", TimestampType(), nullable=False),
])


In [0]:
# =============================================================================
# STEP 8: CONVERT TO SPARK
# =============================================================================
# Create without schema first -- let Spark infer from pandas dtypes.
# Then cast explicitly to ensure all types match the register schema.
# This avoids schema enforcement failures caused by mixed-type columns
# that pandas reports as object dtype.

sdf_register = spark.createDataFrame(pdf_register)

sdf_register = (
    sdf_register
    .withColumn("lat",                 F.col("lat").cast("double"))
    .withColumn("long",                F.col("long").cast("double"))
    .withColumn("easting",             F.col("easting").cast("double"))
    .withColumn("northing",            F.col("northing").cast("double"))
    .withColumn("datum",               F.col("datum").cast("double"))
    .withColumn("period",              F.col("period").cast("string"))
    .withColumn("is_active",           F.col("is_active").cast("boolean"))
    .withColumn("first_seen",          F.col("first_seen").cast("timestamp"))
    .withColumn("register_updated_at", F.col("register_updated_at").cast("timestamp"))
)


In [0]:
# =============================================================================
# STEP 9: WRITE TO REGISTER (CREATE OR APPEND)
# =============================================================================
# First run: create the table.
# Subsequent runs: append new rows only. Existing rows are never touched --
# first_seen is preserved and enrichment is not re-fetched.

if not register_exists:
    print(f"Creating register table: {FULL_REGISTER_NAME}")

    (
        sdf_register
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(FULL_REGISTER_NAME)
    )

    spark.sql(f"""
        COMMENT ON TABLE {FULL_REGISTER_NAME} IS
        'EA real-time measure register. One row per distinct measure URI.
         Built from ea_rt_readings_bronze. Station fields enriched from the
         EA Flood Monitoring stations API (one call per station).
         first_seen records when the measure first appeared in the archive.
         is_active = False marks measures absent from recent archive files.'
    """)

    print("Register created.")

else:
    print(f"Appending {len(pdf_register):,} new measures to {FULL_REGISTER_NAME}")

    (
        sdf_register
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(FULL_REGISTER_NAME)
    )

    print("Append complete.")


In [0]:
# =============================================================================
# STEP 10: SUMMARY
# =============================================================================

spark.sql(f"""
    SELECT
        parameter,
        COUNT(*)                     AS measure_count,
        COUNT(DISTINCT station_uri)  AS station_count,
        SUM(CASE WHEN lat IS NULL THEN 1 ELSE 0 END) AS missing_coords
    FROM {FULL_REGISTER_NAME}
    GROUP BY parameter
    ORDER BY measure_count DESC
""").show(truncate=False)

total = spark.sql(
    f"SELECT COUNT(*) AS n FROM {FULL_REGISTER_NAME}"
).collect()[0]["n"]

print(f"Total measures in register: {total:,}")


In [0]:
# =============================================================================
# SIGNAL COMPLETION
# =============================================================================

print("Measure register complete.")
dbutils.notebook.exit("success")